In [77]:
import os
import re
import fnmatch
import pandas as pd
import time
import numpy as np

graph_folder = "../data/parquet"
all_files = os.listdir(graph_folder)
subgraph_files = fnmatch.filter(all_files, "partial_dynamic_graph_*.parquet")

# Sort files based on their numeric part:
subgraph_files.sort(key=lambda x: int(re.search(r'(\d+)', x).group(0)))

strat_time=time.time()
# Read each file and append to a list

# list_of_dataframes = []
start_time=time.time()
start_time1=time.time()
rel_count = 0
for idx, file in enumerate(subgraph_files):
    df = pd.read_parquet(os.path.join(graph_folder, file), engine='pyarrow')
    index_max_1 = max(df['v1'])
    index_max_2 = max(df['v2'])
    index_max = max(index_max_1, index_max_2)
    if index_max == 123128:
        print("Index is from 1")
        break
    print("Length of df is ",len(df))
    rel_count += len(df)
    # list_of_dataframes.append(df)
    if idx%20==0 and idx!=0:
        print(f"Processed {idx}/{len(subgraph_files)}: {file}; {time.time()-start_time}")
        start_time=time.time()

print("Number of relationship: ", rel_count)
    
print(f"finish: {time.time()-start_time1}")
# Concatenate all dataframes into one
strat_time=time.time()
# full_dynamic_graph = pd.concat(list_of_dataframes, ignore_index=True)
print(f"finish: {time.time()-start_time}")

Length of df is  3
Length of df is  114
Length of df is  3
Length of df is  1
Length of df is  121
Length of df is  120
Length of df is  51
Length of df is  1132
Length of df is  1
Length of df is  108
Length of df is  9
Length of df is  6514
Length of df is  162083
Length of df is  5046
Length of df is  3451
Length of df is  3726
Length of df is  261627
Length of df is  255012
Length of df is  2928388
Length of df is  4374385
Length of df is  2683744
Processed 20/375: partial_dynamic_graph_022.parquet; 2.9196133613586426
Length of df is  667250
Length of df is  4302669
Length of df is  334034
Length of df is  5792449
Length of df is  6245734
Length of df is  2498245
Length of df is  3063869
Length of df is  821373
Length of df is  2456784
Length of df is  3610350
Length of df is  3176087
Length of df is  3607396
Length of df is  4403644
Length of df is  1669828
Length of df is  3930010
Length of df is  4047128
Length of df is  4464699
Length of df is  2396944
Length of df is  2221809


In [69]:
import pandas as pd
from datetime import date
import matplotlib.pyplot as plt

In [70]:
import pandas as pd

entity_list = pd.read_csv('../data/full_concepts.txt')
entity_list

,concept
0,deep neural network
1,convolutional neural network
2,monte carlo simulation
3,cosmic microwave background
4,active galactic nucleus
...,...
123123,liquid mirror
123124,kondo resistivity
123125,diffusional relaxation
123126,alpha_s measurement


In [71]:

import os
from graphdatascience import GraphDataScience

In [72]:
from dotenv import load_dotenv
load_dotenv() 

True

In [73]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_DB = os.environ.get("NEO4J_DB", "neo4j")

# Set up authentication - handle different scenarios
NEO4J_USER = os.environ.get("NEO4J_USER")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")

if NEO4J_USER and NEO4J_PASSWORD:
    NEO4J_AUTH = (NEO4J_USER, NEO4J_PASSWORD)
    print(f"Using authentication with user: {NEO4J_USER}")
else:
    # For local development or Neo4j without authentication
    NEO4J_AUTH = None
    print("No authentication credentials found - attempting connection without auth")

# Test the connection
try:
    gds = GraphDataScience(NEO4J_URI, auth=NEO4J_AUTH)
    # Try to get server info to test connection
    # server_version = gds.version()
    # print(f"Successfully connected to Neo4j. Server version: {server_version}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Please check your Neo4j configuration and credentials.")

Using authentication with user: neo4j


In [74]:
# Load entity_list data into Neo4j as nodes
from neo4j import GraphDatabase

# Create a direct Neo4j driver connection for data loading
print(f"Connecting to Neo4j at: {NEO4J_URI}")
print(f"Database: {NEO4J_DB}")
print(f"Auth method: {'With credentials' if NEO4J_AUTH else 'No authentication'}")

try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    
    if driver == None:
        raise ConnectionError("Problem initializing drive!")
    # Test the connection
    with driver.session(database=NEO4J_DB) as session:
        result = session.run("RETURN 1 as test")
        test_record = result.single()
        if test_record == None:
            raise ValueError("Don't receive anything in test. Should have received 1!")
        
        test_result = test_record["test"]
        
        print(f"Connection test successful: {test_result}")
        print(f"Connected to database: {NEO4J_DB}")
        
except Exception as e:
    print(f"Failed to connect to Neo4j: {e}")
    print("\nTroubleshooting tips:")
    print("1. Make sure Neo4j is running")
    print("2. Check if NEO4J_URI is correct")
    print("3. Verify NEO4J_USER and NEO4J_PASSWORD in your .env file")
    print("4. For local Neo4j without auth, make sure auth is disabled in neo4j.conf")
    raise


Connecting to Neo4j at: bolt://localhost:7687
Database: neo4j
Auth method: With credentials
Connection test successful: 1
Connected to database: neo4j


In [75]:
def clear_database():
    """Clear all nodes and relationships from the Neo4j database"""
    
    if driver == None:
        raise ConnectionError("Problem initializing driver!")
    
    with driver.session(database=NEO4J_DB) as session:
        print("Clearing all relationships and nodes from the database...")
        
        # First, delete all relationships
        print("Step 1: Deleting all relationships...")
        result = session.run("MATCH ()-[r]-() RETURN count(r) as rel_count")
        rel_count = result.single()["rel_count"]
        print(f"Found {rel_count} relationships to delete")
        
        if rel_count > 0:
            session.run("MATCH ()-[r]-() DELETE r")
            print("All relationships deleted")
        
        # Then, delete all nodes
        print("Step 2: Deleting all nodes...")
        result = session.run("MATCH (n) RETURN count(n) as node_count")
        node_count = result.single()["node_count"]
        print(f"Found {node_count} nodes to delete")
        
        if node_count > 0:
            session.run("MATCH (n) DELETE n")
            print("All nodes deleted")
        
        # Drop all indexes to clean up completely
        print("Step 3: Dropping all indexes...")
        try:
            session.run("DROP INDEX entity_index_idx IF EXISTS")
            session.run("DROP INDEX entity_description_idx IF EXISTS")
            print("Indexes dropped")
        except Exception as e:
            print(f"Note: Some indexes might not exist: {e}")
        
        # Verify the database is empty
        print("Step 4: Verifying database is empty...")
        result = session.run("MATCH (n) RETURN count(n) as remaining_nodes")
        remaining_nodes = result.single()["remaining_nodes"]
        
        result = session.run("MATCH ()-[r]-() RETURN count(r) as remaining_rels")
        remaining_rels = result.single()["remaining_rels"]
        
        if remaining_nodes == 0 and remaining_rels == 0:
            print("✅ Database successfully cleared!")
            print("✅ 0 nodes remaining")
            print("✅ 0 relationships remaining")
        else:
            print(f"⚠️  Warning: {remaining_nodes} nodes and {remaining_rels} relationships still remain")

# Call the function to clear the database
clear_database()

Clearing all relationships and nodes from the database...
Step 1: Deleting all relationships...
Found 1224246 relationships to delete
All relationships deleted
Step 2: Deleting all nodes...
Found 123128 nodes to delete
All nodes deleted
Step 3: Dropping all indexes...
Indexes dropped
Step 4: Verifying database is empty...
✅ Database successfully cleared!
✅ 0 nodes remaining
✅ 0 relationships remaining


In [76]:

def load_entities_to_neo4j(entity_df):
    """Load entities from DataFrame to Neo4j as nodes"""
    
    # Prepare the data with index as a property
    entities_data = []
    for idx, row in entity_df.iterrows():
        entities_data.append({
            'index': int(idx),
            'description': row['concept']
        })
    
    # Cypher query to create nodes
    cypher_query = """
    UNWIND $entities AS entity
    CREATE (n:Entity {
        index: entity.index,
        description: entity.description
    })
    """
    
    # Execute in batches for better performance
    batch_size = 1000
    if driver == None:
        raise ConnectionError("Problem initializing drive!")
    with driver.session(database=NEO4J_DB) as session:
        # Clear existing Entity nodes first (optional)
        print("Clearing existing Entity nodes...")
        session.run("MATCH (n:Entity) DELETE n")
        
        # Load data in batches
        total_entities = len(entities_data)
        for i in range(0, total_entities, batch_size):
            batch = entities_data[i:i + batch_size]
            session.run(cypher_query, entities=batch)
            print(f"Loaded batch {i//batch_size + 1}: {len(batch)} entities (Total progress: {min(i + batch_size, total_entities)}/{total_entities})")
    
    print(f"Successfully loaded {total_entities} entities to Neo4j!")

# Load the entities
print(f"\nLoading {len(entity_list)} entities to Neo4j...")
load_entities_to_neo4j(entity_list)


Loading 123128 entities to Neo4j...
Clearing existing Entity nodes...
Loaded batch 1: 1000 entities (Total progress: 1000/123128)
Loaded batch 2: 1000 entities (Total progress: 2000/123128)
Loaded batch 3: 1000 entities (Total progress: 3000/123128)
Loaded batch 4: 1000 entities (Total progress: 4000/123128)
Loaded batch 5: 1000 entities (Total progress: 5000/123128)
Loaded batch 6: 1000 entities (Total progress: 6000/123128)
Loaded batch 7: 1000 entities (Total progress: 7000/123128)
Loaded batch 8: 1000 entities (Total progress: 8000/123128)
Loaded batch 9: 1000 entities (Total progress: 9000/123128)
Loaded batch 10: 1000 entities (Total progress: 10000/123128)
Loaded batch 11: 1000 entities (Total progress: 11000/123128)
Loaded batch 12: 1000 entities (Total progress: 12000/123128)
Loaded batch 13: 1000 entities (Total progress: 13000/123128)
Loaded batch 14: 1000 entities (Total progress: 14000/123128)
Loaded batch 15: 1000 entities (Total progress: 15000/123128)
Loaded batch 16: 

In [57]:
# Create indexes for better performance and verify the data
with driver.session(database=NEO4J_DB) as session:
    # Create index on the index property for fast lookups
    print("Creating index on Entity.index...")
    session.run("CREATE INDEX entity_index_idx IF NOT EXISTS FOR (n:Entity) ON (n.index)")
    
    # Create index on description for text searches
    print("Creating index on Entity.description...")
    session.run("CREATE INDEX entity_description_idx IF NOT EXISTS FOR (n:Entity) ON (n.description)")
    
    # Verify the data was loaded correctly
    print("\nVerifying loaded data:")
    result = session.run("MATCH (n:Entity) RETURN count(n) as total_count")
    total_count = result.single()["total_count"]
    print(f"Total Entity nodes in Neo4j: {total_count}")
    
    # Show some sample entities
    print("\nSample entities:")
    result = session.run("""
        MATCH (n:Entity) 
        RETURN n.index as index, n.description as description 
        ORDER BY n.index 
        LIMIT 5
    """)
    for record in result:
        print(f"Index: {record['index']}, Description: '{record['description']}'")
    
    print("\nLast 5 entities:")
    result = session.run("""
        MATCH (n:Entity) 
        RETURN n.index as index, n.description as description 
        ORDER BY n.index DESC 
        LIMIT 5
    """)
    for record in result:
        print(f"Index: {record['index']}, Description: '{record['description']}'")

print("\nEntity loading completed successfully!")

Creating index on Entity.index...
Creating index on Entity.description...

Verifying loaded data:
Total Entity nodes in Neo4j: 123128

Sample entities:
Index: 0, Description: 'deep neural network'
Index: 1, Description: 'convolutional neural network'
Index: 2, Description: 'monte carlo simulation'
Index: 3, Description: 'cosmic microwave background'
Index: 4, Description: 'active galactic nucleus'

Last 5 entities:
Index: 123127, Description: 'center gauge'
Index: 123126, Description: 'alpha_s measurement'
Index: 123125, Description: 'diffusional relaxation'
Index: 123124, Description: 'kondo resistivity'
Index: 123123, Description: 'liquid mirror'

Entity loading completed successfully!


### Load relationship

In [58]:
from tqdm import tqdm

In [ ]:
def filter_relationship(data_path):
    important_coeff = 0.95
    df = pd.read_parquet(data_path)
    print(len(df))
    date_origin = date(1990,1,1)
    
    date_2012 = date(2012,1,1)
    
    df['citation_balanced'] = 0
    for i, year in enumerate(range(2012,2024)):
        df['citation_balanced'] += df[f'c{year}']*important_coeff**(12-i)
    df['time_from_2012'] = df['time'].map(lambda x: x + (date_origin - date_2012).days)
    df.drop(columns='time', inplace=True)
    df = df[((df['time_from_2012'] > 0) | (df['ct'] > 15))]
    return df

with driver.session(database=NEO4J_DB) as session:
    graph_folder = "../data/parquet"
    all_files = os.listdir(graph_folder)
    subgraph_files = fnmatch.filter(all_files, "partial_dynamic_graph_*.parquet")

    # Sort files based on their numeric part:
    subgraph_files.sort(key=lambda x: int(re.search(r'(\d+)', x).group(0)))
    for file in tqdm(subgraph_files):
        data_path = os.path.join(graph_folder, file)
        print(f"Processing file: {file}")
        df = filter_relationship(data_path)
        
        for idx, row in df.iterrows():
            # Extract all properties except source and target
            row_props = {k: v for k, v in row.items() if k not in ['v1', 'v2']}
            
            cypher_query = """
            MATCH (a:Entity {index: $source_index}), (b:Entity {index: $target_index})
            
            // Check if MAX_BALANCED_CITATION edge exists
            OPTIONAL MATCH (a)-[max_edge:MAX_BALANCED_CITATION]->(b)
            OPTIONAL MATCH (a)-[total_edge:TOTAL_CITATION]->(b)
            OPTIONAL MATCH (a)-[appear_edge:NO_APPEARANCES]->(b)
            
            WITH a, b, max_edge, total_edge, appear_edge,
                 CASE WHEN max_edge IS NULL THEN true 
                      ELSE $citation_balanced > max_edge.citation_balanced 
                 END as should_update_max
            
            // Handle MAX_BALANCED_CITATION edge
            FOREACH (x IN CASE WHEN max_edge IS NULL THEN [1] ELSE [] END |
                CREATE (a)-[:MAX_BALANCED_CITATION $row_props]->(b)
            )
            FOREACH (x IN CASE WHEN max_edge IS NOT NULL AND should_update_max THEN [1] ELSE [] END |
                DELETE max_edge
            )
            FOREACH (x IN CASE WHEN max_edge IS NOT NULL AND should_update_max THEN [1] ELSE [] END |
                CREATE (a)-[:MAX_BALANCED_CITATION $row_props]->(b)
            )
            
            // Handle TOTAL_CITATION edge
            FOREACH (x IN CASE WHEN total_edge IS NULL THEN [1] ELSE [] END |
                CREATE (a)-[:TOTAL_CITATION $row_props]->(b)
            )
            FOREACH (x IN CASE WHEN total_edge IS NOT NULL THEN [1] ELSE [] END |
                SET total_edge += $row_props
            )
            
            // Handle NO_APPEARANCES edge
            FOREACH (x IN CASE WHEN appear_edge IS NULL THEN [1] ELSE [] END |
                CREATE (a)-[:NO_APPEARANCES {count: 1}]->(b)
            )
            FOREACH (x IN CASE WHEN appear_edge IS NOT NULL THEN [1] ELSE [] END |
                SET appear_edge.count = appear_edge.count + 1
            )
            """
            
            # Prepare parameters
            params = {
                'source_index': int(row['v1']),
                'target_index': int(row['v2']),
                'row_props': {k: (int(v) if isinstance(v, (int, np.integer)) else 
                                float(v) if isinstance(v, (float, np.floating)) else v) 
                            for k, v in row_props.items()},
                'citation_balanced': float(row['citation_balanced'])
            }
            
            session.run(cypher_query, **params)
            
            if idx % 10000 == 0:
                print(f"Processed {idx} relationships in {file}")

  0%|          | 0/375 [00:00<?, ?it/s]

Processing file: partial_dynamic_graph_000.parquet
3
1990-01-01
2012-01-01
Processing file: partial_dynamic_graph_001.parquet
114
1990-01-01
2012-01-01


  1%|          | 4/375 [00:01<02:25,  2.54it/s]

Processing file: partial_dynamic_graph_002.parquet
3
1990-01-01
2012-01-01
Processing file: partial_dynamic_graph_005.parquet
1
1990-01-01
2012-01-01
Processed 0 relationships in partial_dynamic_graph_005.parquet
Processing file: partial_dynamic_graph_006.parquet
121
1990-01-01
2012-01-01
Processed 0 relationships in partial_dynamic_graph_006.parquet


  1%|▏         | 5/375 [00:02<03:42,  1.66it/s]

...

Processing file: partial_dynamic_graph_017.parquet
3726
1990-01-01
2012-01-01
Processed 0 relationships in partial_dynamic_graph_017.parquet


  4%|▍         | 16/375 [2:03:45<68:37:51, 688.22s/it]

Processing file: partial_dynamic_graph_018.parquet
261627
1990-01-01
2012-01-01
Processed 0 relationships in partial_dynamic_graph_018.parquet
Processed 10000 relationships in partial_dynamic_graph_018.parquet
Processed 20000 relationships in partial_dynamic_graph_018.parquet
Processed 30000 relationships in partial_dynamic_graph_018.parquet
Processed 50000 relationships in partial_dynamic_graph_018.parquet
Processed 60000 relationships in partial_dynamic_graph_018.parquet
Processed 70000 relationships in partial_dynamic_graph_018.parquet


  4%|▍         | 16/375 [2:12:17<49:28:22, 496.11s/it]


DatabaseUnavailable: {code: Neo.TransientError.General.DatabaseUnavailable} {message: The database is not currently available to serve your request, refer to the database logs for more details. Retrying your request at a later time may succeed.}